# Viz — Bokeh (WebGL)

Large scatter using Bokeh's WebGL backend. PNG export uses ``export_png`` (may require a headless browser driver such as Selenium + Chrome/Chromium, or use ``bokeh.io.show`` in a live session).

**WebGL canvas in the notebook**

Subsamples vertices for responsiveness, maps orthographic disk coordinates (inside the unit circle), and colors by `log1p(degree)` on a **1400×1400** pixel figure. **Interpretation:** orthographic and stereo views differ; compare to other viz notebooks only when you intentionally match projections.

In [ ]:
from pathlib import Path

import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource
from bokeh.io import export_png

import dmercator_io as dm

output_notebook()

merged = dm.load_merged_parquet(Path("cache") / "merged.parquet")
u, v = dm.ortho_xy_disk(merged)
rng = np.random.default_rng(3)
n = min(120_000, len(merged))
ix = rng.choice(len(merged), size=n, replace=False)
source = ColumnDataSource(
    data=dict(
        u=u[ix],
        v=v[ix],
        deg=np.log1p(merged["degree"].to_numpy()[ix]),
    )
)

p = figure(
    width=1400,
    height=1400,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    title="Ortho disk (WebGL) colored by log1p(degree)",
    output_backend="webgl",
    match_aspect=True,
)
p.circle("u", "v", source=source, size=2, alpha=0.25, color="navy")
show(p)


**Optional PNG export**

Uses Bokeh’s `export_png`, which often needs extra system setup (browser driver / headless dependencies). If this raises, rely on **Save** in the toolbar in a live session or capture the notebook output instead.

In [ ]:
# Optional static export (driver-dependent)
out = Path("cache") / "figures"
out.mkdir(parents=True, exist_ok=True)
try:
    export_png(p, filename=out / "bokeh_ortho.png")
    print("Wrote", (out / "bokeh_ortho.png").resolve())
except Exception as e:
    print("export_png skipped:", e)
